# Cell embeddings

This notebook is the entity-level tutorial for single-cell state embeddings. `BioEmbedder.embed(...)` routes cell models through the single-cell preprocessing policy, stores embeddings in `.obsm`, and records provenance in `.uns`.

The key point: cell embeddings are observation-level outputs. They belong in `.obsm`, unlike perturbation/action embeddings whose canonical table lives in `.uns`.


In [ ]:
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd

from embpy import BioEmbedder, pl, tl

RUN_REAL_EMBEDDING = False
RANDOM_STATE = 7
rng = np.random.default_rng(RANDOM_STATE)

embedder = BioEmbedder(device="auto", organism="human")

cell_types = np.array(["T", "T", "B", "B", "myeloid", "myeloid", "epithelial", "epithelial"])
batches = np.array(["batch1", "batch2", "batch1", "batch2", "batch1", "batch2", "batch1", "batch2"])
gene_names = [f"Gene{i}" for i in range(40)]
counts = rng.poisson(lam=2.0, size=(len(cell_types), len(gene_names))).astype("float32")

adata = ad.AnnData(
    X=counts,
    obs=pd.DataFrame({"cell_type": cell_types, "batch": batches}, index=[f"cell_{i}" for i in range(len(cell_types))]),
    var=pd.DataFrame(index=gene_names),
)
adata.layers["counts"] = adata.X.copy()
adata


## 1. Preprocessing-aware cell embeddings

`preprocessing="auto"` asks embpy to inspect the requested model policy and prepare the layer it expects. Examples:

- PCA can use log-normalized highly variable genes.
- Geneformer/scGPT-style models expect model-specific tokenization and vocab matching.
- STATE/STACK use their own Arc Institute wrappers and checkpoint options.

You can override this with `preprocessing="raw"`, `"standard"`, or `"none"`, plus options such as `select_hvg`, `n_top_genes`, `target_sum`, `log_transform`, and `scale`.


In [ ]:
if RUN_REAL_EMBEDDING:
    pca_cells = embedder.embed(
        adata,
        entity_type="cell",
        model="pca",
        output="anndata",
        preprocessing="auto",
        n_pca_components=16,
        select_hvg=True,
        n_top_genes=20,
        key="X_cell_pca",
    )
    print(pca_cells.obsm["X_cell_pca"].shape)
else:
    print("Set RUN_REAL_EMBEDDING=True to compute PCA cell embeddings through BioEmbedder.embed(...).")


## 2. Heavy single-cell foundation models

The same public API covers scGPT, Geneformer, STATE, STACK, and related wrappers when their pixi environments and checkpoints are available.


In [ ]:
RUN_HEAVY_MODELS = False

if RUN_HEAVY_MODELS:
    scgpt_cells = embedder.embed(
        adata,
        entity_type="cell",
        model="scgpt",
        output="anndata",
        preprocessing="auto",
        batch_size=16,
        key="X_scgpt",
    )

    stack_cells = embedder.embed(
        adata,
        entity_type="cell",
        model="stack",
        output="anndata",
        preprocessing="raw",
        stack_checkpoint="/path/to/bc_large.ckpt",
        stack_genelist="/path/to/basecount_1000per_15000max.pkl",
        key="X_stack",
    )


## 3. Plot-ready cell AnnData


In [ ]:
def make_cell_demo(source: ad.AnnData) -> ad.AnnData:
    out = source.copy()
    centers = {ct: rng.normal(size=12) for ct in sorted(set(source.obs["cell_type"]))}
    X_pca = np.vstack([centers[ct] + rng.normal(scale=0.2, size=12) for ct in source.obs["cell_type"]]).astype("float32")
    X_stack = (X_pca @ rng.normal(size=(12, 10)) + rng.normal(scale=0.25, size=(source.n_obs, 10))).astype("float32")
    X_scgpt = (X_pca @ rng.normal(size=(12, 8)) + rng.normal(scale=0.3, size=(source.n_obs, 8))).astype("float32")
    out.obsm["X_pca"] = X_pca
    out.obsm["X_stack"] = X_stack
    out.obsm["X_scgpt"] = X_scgpt
    out.uns["embeddings"] = {
        "X_pca": {"entity_type": "cell", "storage": "obsm", "model": "pca"},
        "X_stack": {"entity_type": "cell", "storage": "obsm", "model": "stack"},
        "X_scgpt": {"entity_type": "cell", "storage": "obsm", "model": "scgpt"},
    }
    return out

cell_space = make_cell_demo(adata)
cell_space


## 4. Plot cell-state embeddings by annotations


In [ ]:
pl.plot_embedding_space(
    cell_space,
    obsm_key="X_stack",
    method="pca",
    color="cell_type",
    annotate=True,
    title="Cell state embedding by cell type",
)

pl.embedding_color_panel(
    cell_space,
    obsm_key="X_stack",
    method="pca",
    color_keys=["cell_type", "batch"],
    title="Cell embedding annotations",
)


## 5. Compare cell embedding models


In [ ]:
_, mean_overlap = tl.compute_knn_overlap(cell_space, "X_stack", "X_scgpt", k=3)
print(f"Mean STACK/scGPT KNN overlap: {mean_overlap:.3f}")

pl.knn_overlap(cell_space, obsm_keys=["X_pca", "X_stack", "X_scgpt"], k=3)
pl.cross_embedding_correlation(cell_space, "X_stack", "X_scgpt")
pl.embedding_norms(cell_space, obsm_keys=["X_pca", "X_stack", "X_scgpt"])


## 6. Attach action embeddings for world models

World-model training reads cell/state embeddings from `.obsm[data.state_obsm_key]` and action embeddings from `.obsm[action_embedding.obsm_key]`. Keep the canonical perturbation payload in `.uns`, then materialize `.obsm` only when training needs per-cell action vectors.


In [ ]:
if RUN_REAL_EMBEDDING:
    from embpy.io.exporters import materialize_perturbation_obsm

    cell_space.obs["perturbation"] = ["TP53", "TP53", "EGFR", "EGFR", "MYC", "MYC", "control", "control"]
    cell_space = embedder.embed(
        ["TP53", "EGFR", "MYC"],
        entity_type="gene",
        id_type="symbol",
        model="esm2_650M",
        output="anndata",
        target=cell_space,
        attach_to="uns",
        key="X_pert_esm2_650M",
    )
    materialize_perturbation_obsm(
        cell_space,
        embedding_key="X_pert_esm2_650M",
        perturbation_key="perturbation",
        obsm_key="X_pert_esm2_650M",
        missing="zero",
    )
